In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score

print("Libraries imported successfully")

In [ ]:
# Load preprocessed data
data_path = Path.cwd().parent / "shared" / "output" / "production_preprocessed_data.csv"
df = pd.read_csv(data_path, encoding='utf-8-sig')

print(f"Loaded {len(df)} rows from preprocessed data")
print(f"Participants: {df['prolific_id'].nunique()}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nPhonation distribution:")
print(df['phonation'].value_counts())

In [ ]:
# ================ LDA 분석 ================

print("=" * 60)
print("Linear Discriminant Analysis (LDA)")
print("=" * 60)

# 데이터 준비 (결측치 제거)
lda_data = df[['normed_scaled_vot', 'normed_f0', 'phonation']].dropna()
X = lda_data[['normed_scaled_vot', 'normed_f0']].values
y = lda_data['phonation'].values

print(f"\n데이터 크기: {len(X)} observations")
print(f"클래스 분포:")
for phonation in ['lenis', 'tense', 'aspirated']:
    count = (y == phonation).sum()
    print(f"  {phonation}: {count} ({count/len(y)*100:.1f}%)")

# LDA 모델 학습
lda = LinearDiscriminantAnalysis()
lda.fit(X, y)

# LD 점수 계산 (transformed coordinates)
X_lda = lda.transform(X)

# 예측 및 정확도
y_pred = lda.predict(X)
accuracy = accuracy_score(y, y_pred)

print("\n" + "=" * 60)
print("분류 성능")
print("=" * 60)
print(f"\n전체 정확도: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Cross-validation
cv_scores = cross_val_score(lda, X, y, cv=5)
print(f"5-fold CV 정확도: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

# Confusion Matrix
print("\n" + "-" * 60)
print("Confusion Matrix")
print("-" * 60)
cm = confusion_matrix(y, y_pred, labels=['lenis', 'tense', 'aspirated'])
cm_df = pd.DataFrame(cm, 
                     index=['True: lenis', 'True: tense', 'True: aspirated'],
                     columns=['Pred: lenis', 'Pred: tense', 'Pred: aspirated'])
print(cm_df)

# 카테고리별 정확도
print("\n" + "-" * 60)
print("카테고리별 분류 성능")
print("-" * 60)
print(classification_report(y, y_pred, target_names=['lenis', 'tense', 'aspirated']))

# LDA Coefficients (변수 기여도)
print("\n" + "=" * 60)
print("LDA Coefficients (변수 기여도)")
print("=" * 60)
print("\nLD1 (첫 번째 판별축):")
print(f"  Normed Scaled VOT: {lda.coef_[0, 0]:.4f}")
print(f"  Normed F0: {lda.coef_[0, 1]:.4f}")

if lda.coef_.shape[0] > 1:
    print("\nLD2 (두 번째 판별축):")
    print(f"  Normed Scaled VOT: {lda.coef_[1, 0]:.4f}")
    print(f"  Normed F0: {lda.coef_[1, 1]:.4f}")

# Explained variance ratio
print("\n" + "-" * 60)
print("Explained Variance Ratio (축별 설명력)")
print("-" * 60)
explained_var = lda.explained_variance_ratio_
for i, var in enumerate(explained_var):
    print(f"LD{i+1}: {var:.4f} ({var*100:.2f}%)")

# LD 점수를 DataFrame에 추가
lda_results = lda_data.copy()
lda_results['LD1'] = X_lda[:, 0]
if X_lda.shape[1] > 1:
    lda_results['LD2'] = X_lda[:, 1]
lda_results['predicted'] = y_pred
lda_results['correct'] = (y == y_pred)

# 각 phonation의 LD1 평균
print("\n" + "=" * 60)
print("Phonation별 LD1 평균 (중심점)")
print("=" * 60)
for phonation in ['lenis', 'tense', 'aspirated']:
    subset = lda_results[lda_results['phonation'] == phonation]
    mean_ld1 = subset['LD1'].mean()
    std_ld1 = subset['LD1'].std()
    print(f"{phonation.capitalize()}: Mean = {mean_ld1:.4f}, SD = {std_ld1:.4f}")

# LD1 축상의 거리
print("\n" + "-" * 60)
print("LD1 축상의 Phonation 간 거리")
print("-" * 60)
ld1_means = {}
for phonation in ['lenis', 'tense', 'aspirated']:
    ld1_means[phonation] = lda_results[lda_results['phonation'] == phonation]['LD1'].mean()

for phon1, phon2 in combinations(['lenis', 'tense', 'aspirated'], 2):
    dist = abs(ld1_means[phon1] - ld1_means[phon2])
    print(f"{phon1.capitalize()} vs {phon2.capitalize()}: {dist:.4f}")

print("\n" + "=" * 60)
print("분석 완료")
print("=" * 60)
print(f"\nLDA 결과 DataFrame: lda_results ({len(lda_results)} rows)")
print(f"컬럼: {list(lda_results.columns)}")

# 결과 저장
output_dir = Path.cwd().parent / "shared" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "production_individual_lda_results.csv"
lda_results.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\n저장 완료: {output_path}")